## 2. Data Understanding & Cleaning

This notebook examines the structure, quality, and characteristics of the dataset used in the analysis and prepares it for exploratory analysis and predictive modeling.

## 2.1 Data Source

The dataset used in this project is the Diabetes 130-US Hospitals for Years 1999–2008 dataset from the UCI Machine Learning Repository. It contains 101,766 inpatient encounters collected from 130 U.S. hospitals and integrated delivery networks between 1999 and 2008. The dataset includes patient demographics, admission and discharge information, diagnoses, laboratory procedures, medications, and prior healthcare utilization.

Each row represents a hospital encounter, not a unique patient; therefore, a patient may appear in multiple records.

The `readmitted` variable classifies each encounter into three outcomes:

- `<30` — readmitted within 30 days
- `>30` — readmitted after 30 days
- `NO` — no recorded readmission

For this project, 30-day readmission (`<30`) is the outcome of interest.



## 2.2 Data Loading and Initial Inspection

The raw dataset is loaded without modification to examine its dimensions, structure, and data types.

### 2.2.1 Load the Dataset

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/diabetic_data.csv")

df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


### 2.2.2 Dataset Dimensions

In [2]:
# Check dataset dimensions
df.shape

(101766, 50)

The raw dataset contains 101,766 hospital encounters and 50 variables.

### 2.2.3 Column Overview

In [4]:
# Display column names
df.columns.tolist()

['encounter_id',
 'patient_nbr',
 'race',
 'gender',
 'age',
 'weight',
 'admission_type_id',
 'discharge_disposition_id',
 'admission_source_id',
 'time_in_hospital',
 'payer_code',
 'medical_specialty',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'diag_1',
 'diag_2',
 'diag_3',
 'number_diagnoses',
 'max_glu_serum',
 'A1Cresult',
 'metformin',
 'repaglinide',
 'nateglinide',
 'chlorpropamide',
 'glimepiride',
 'acetohexamide',
 'glipizide',
 'glyburide',
 'tolbutamide',
 'pioglitazone',
 'rosiglitazone',
 'acarbose',
 'miglitol',
 'troglitazone',
 'tolazamide',
 'examide',
 'citoglipton',
 'insulin',
 'glyburide-metformin',
 'glipizide-metformin',
 'glimepiride-pioglitazone',
 'metformin-rosiglitazone',
 'metformin-pioglitazone',
 'change',
 'diabetesMed',
 'readmitted']

### 2.2.4 Data Types and Completeness


In [5]:
# Inspect data types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   race                      101766 non-null  str  
 3   gender                    101766 non-null  str  
 4   age                       101766 non-null  str  
 5   weight                    101766 non-null  str  
 6   admission_type_id         101766 non-null  int64
 7   discharge_disposition_id  101766 non-null  int64
 8   admission_source_id       101766 non-null  int64
 9   time_in_hospital          101766 non-null  int64
 10  payer_code                101766 non-null  str  
 11  medical_specialty         101766 non-null  str  
 12  num_lab_procedures        101766 non-null  int64
 13  num_procedures            101766 non-null  int64
 14  num_medications           10176

The dataset contains 13 integer variables and 37 categorical variables. Most columns appear complete based on standard null detection. 
Standard null counts alone are not sufficient to assess completeness. These values will be assessed separately during the data-quality assessment.

### 2.2.5 Unique Patients and Encounters

Because each row represents a hospital encounter, the number of unique patients is compared with the number of encounters to determine the extent to which patients appear multiple times in the dataset.

In [6]:
# Compare hospital encounters with unique patients
n_encounters = df["encounter_id"].nunique()
n_patients = df["patient_nbr"].nunique()

print(f"Unique encounters: {n_encounters:,}")
print(f"Unique patients: {n_patients:,}")
print(f"Repeat encounters: {n_encounters - n_patients:,}")

Unique encounters: 101,766
Unique patients: 71,518
Repeat encounters: 30,248


The dataset contains 101,766 unique hospital encounters involving 71,518 unique patients. This confirms that some patients appear in the dataset across multiple hospital encounters.

The difference of 30,248 between encounter count and unique patient count indicates repeated utilization within the study population. Because repeated encounters from the same patient are not independent, patient-level grouping will need to be considered when splitting the data for predictive modeling to reduce the risk of data leakage.

### 2.2.6 Target Variable Identification

The outcome variable for the analysis is `readmitted`, which records whether an encounter was followed by a hospital readmission within 30 days, after 30 days, or no recorded readmission.

For predictive modeling, the outcome will later be transformed into a binary target representing 30-day readmission:

- `<30` → 1
- `>30` → 0
- `NO` → 0

The transformation will be performed during feature engineering to preserve the original variable during data understanding and exploratory analysis.

## 2.3 Data Quality Assessment

Before cleaning the dataset, data quality is assessed to identify missing values, placeholder values, duplicate records, and other issues that may affect the analysis.